In [ ]:

# ============================================================
# CUSTOMER CHURN CAUSAL INFERENCE ANALYSIS
# Jupyter Notebook / Windows version
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

from dowhy import CausalModel


# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

DATA_PATH = Path("Upload your path")

OUTPUT_DIR = Path("Upload your path")

# Update these paths to match your local machine before running.
# The original dataset is not included in this repository.

OUTCOME = "churn"

TREATMENTS = [
    "international_plan",
    "voice_mail_plan",
    "customer_service_contact",
]

CONFOUNDERS = [
    "total_day_minutes",
    "total_day_calls",
    "total_eve_minutes",
    "total_eve_calls",
    "total_night_minutes",
    "total_night_calls",
]

SENSITIVITY_STRENGTHS = [
    0.01,
    0.05,
    0.10,
    0.20,
    0.30,
]

RANDOM_SEED = 42

NUM_SIMULATIONS = 100


# ============================================================
# 2. DATA PREPROCESSING
# ============================================================

def load_data(data_path):

    if not data_path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {data_path}"
        )

    df = pd.read_csv(data_path)

    required_columns = [
        "churn",
        "international_plan",
        "voice_mail_plan",
        "number_customer_service_calls",
        *CONFOUNDERS,
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    # Convert binary variables
    binary_columns = [
        "churn",
        "international_plan",
        "voice_mail_plan",
    ]

    for col in binary_columns:

        if df[col].dtype == "object":

            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .str.lower()
                .map({
                    "yes": 1,
                    "no": 0,
                    "true": 1,
                    "false": 0,
                    "1": 1,
                    "0": 0,
                })
            )

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # Convert confounders to numeric
    for col in CONFOUNDERS:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # Create customer service contact treatment
    df["customer_service_contact"] = (
        pd.to_numeric(
            df["number_customer_service_calls"],
            errors="coerce"
        ) > 0
    ).astype(float)

    analysis_columns = [
        OUTCOME,
        *TREATMENTS,
        *CONFOUNDERS,
    ]

    # Check missing values
    missing_summary = df[
        analysis_columns
    ].isnull().sum()

    if missing_summary.any():

        print("\nMissing values detected:")

        print(
            missing_summary[
                missing_summary > 0
            ]
        )

        df = df.dropna(
            subset=analysis_columns
        ).copy()

    # Check binary variables
    for col in [OUTCOME, *TREATMENTS]:

        values = set(
            df[col].dropna().unique()
        )

        if not values.issubset({0, 1, 0.0, 1.0}):

            raise ValueError(
                f"{col} must contain only 0 and 1. "
                f"Found: {values}"
            )

    df[analysis_columns] = df[
        analysis_columns
    ].astype(float)

    print("\nDATA PREPROCESSING COMPLETED")
    print("-" * 60)

    print(f"Observations: {len(df)}")
    print(f"Churn rate: {df[OUTCOME].mean():.2%}")

    print("\nMissing values after preprocessing:")
    print(df[analysis_columns].isnull().sum().sum())

    print("\nTreatment distributions:")

    for treatment in TREATMENTS:

        print(
            f"{treatment}: "
            f"{df[treatment].value_counts().to_dict()}"
        )

    return df


# ============================================================
# 3. DAG CONSTRUCTION
# ============================================================

def make_dag(treatment):

    edges = [
        f"{treatment} -> {OUTCOME}"
    ]

    for confounder in CONFOUNDERS:

        edges.append(
            f"{confounder} -> {treatment}"
        )

        edges.append(
            f"{confounder} -> {OUTCOME}"
        )

    dag_edges = ";\n".join(edges)

    dag = f"""
    digraph {{
        {dag_edges};
    }}
    """

    return dag


# ============================================================
# 4. ATE ESTIMATION
# ============================================================

def estimate_ate(df, treatment):

    model = CausalModel(
        data=df,
        treatment=treatment,
        outcome=OUTCOME,
        common_causes=CONFOUNDERS,
        graph=make_dag(treatment),
    )

    identified_estimand = model.identify_effect(
        proceed_when_unidentifiable=True
    )

    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.generalized_linear_model",
        target_units="ate",
        confidence_intervals=True,
        method_params={
            "glm_family": sm.families.Binomial(),
        },
    )

    ate = float(estimate.value)

    ci_lower = np.nan
    ci_upper = np.nan

    try:

        ci = estimate.get_confidence_intervals()

        ci_values = np.asarray(
            ci,
            dtype=float
        ).reshape(-1)

        if len(ci_values) >= 2:

            ci_lower = float(ci_values[0])
            ci_upper = float(ci_values[1])

    except Exception as error:

        warnings.warn(
            f"Could not extract CI for {treatment}: {error}"
        )

    return (
        model,
        identified_estimand,
        estimate,
        ate,
        ci_lower,
        ci_upper,
    )


# ============================================================
# 5. REFUTATION TESTS
# ============================================================

def run_refutation_tests(
    model,
    identified_estimand,
    estimate,
):

    results = {}

    # --------------------------------------------------------
    # Placebo treatment refutation
    # --------------------------------------------------------

    try:

        placebo = model.refute_estimate(
            identified_estimand,
            estimate,
            method_name="placebo_treatment_refuter",
            placebo_type="permute",
            num_simulations=NUM_SIMULATIONS,
        )

        results["placebo_result"] = str(placebo)

        results["placebo_p_value"] = getattr(
            placebo,
            "p_value",
            np.nan
        )

    except Exception as error:

        results["placebo_result"] = str(error)

        results["placebo_p_value"] = np.nan

    # --------------------------------------------------------
    # Random common cause refutation
    # --------------------------------------------------------

    try:

        random_cause = model.refute_estimate(
            identified_estimand,
            estimate,
            method_name="random_common_cause",
            num_simulations=NUM_SIMULATIONS,
        )

        results["random_common_cause_result"] = str(
            random_cause
        )

        results["random_common_cause_p_value"] = getattr(
            random_cause,
            "p_value",
            np.nan
        )

    except Exception as error:

        results["random_common_cause_result"] = str(error)

        results["random_common_cause_p_value"] = np.nan

    return results


# ============================================================
# 6. UNOBSERVED CONFOUNDING SENSITIVITY ANALYSIS
# ============================================================

def extract_scalar_effect(refutation):

    new_effect = getattr(
        refutation,
        "new_effect",
        None
    )

    if new_effect is None:
        return np.nan

    try:

        values = np.asarray(
            new_effect,
            dtype=float
        ).reshape(-1)

        if values.size == 1:
            return float(values[0])

        warnings.warn(
            "Refutation new_effect is not a scalar."
        )

        return np.nan

    except (TypeError, ValueError):

        return np.nan


def run_sensitivity_analysis(
    model,
    identified_estimand,
    estimate,
):

    sensitivity_results = []

    original_ate = float(estimate.value)

    for strength in SENSITIVITY_STRENGTHS:

        result_row = {
            "strength": strength,
            "original_ate": original_ate,
            "new_ate": np.nan,
            "status": "FAILED",
            "result": None,
        }

        try:

            refutation = model.refute_estimate(
                identified_estimand,
                estimate,
                method_name="add_unobserved_common_cause",
                confounders_effect_on_treatment="binary_flip",
                confounders_effect_on_outcome="binary_flip",
                effect_strength_on_treatment=strength,
                effect_strength_on_outcome=strength,
                show_progress_bar=False,
            )

            result_row["new_ate"] = extract_scalar_effect(
                refutation
            )

            result_row["status"] = "SUCCESS"

            result_row["result"] = str(refutation)

        except Exception as error:

            result_row["result"] = (
                f"ERROR: {error}"
            )

        sensitivity_results.append(result_row)

        print(
            f"Strength: {strength:.2f} | "
            f"Original ATE: {original_ate:.6f} | "
            f"New ATE: {result_row['new_ate']} | "
            f"Status: {result_row['status']}"
        )

    return sensitivity_results


# ============================================================
# 7. MAIN ANALYSIS
# ============================================================

def run_analysis(df):

    all_results = []

    all_sensitivity_results = []

    for treatment in TREATMENTS:

        print("\n" + "=" * 65)
        print(f"ANALYZING TREATMENT: {treatment}")
        print("=" * 65)

        (
            model,
            identified_estimand,
            estimate,
            ate,
            ci_lower,
            ci_upper,
        ) = estimate_ate(
            df,
            treatment
        )

        print(f"\nATE: {ate:.6f}")

        print(
            f"95% CI: ({ci_lower:.6f}, {ci_upper:.6f})"
        )

        result_row = {
            "treatment": treatment,
            "ate": ate,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "ate_percentage_points": ate * 100,
            "n_observations": len(df),
        }

        # Refutation tests
        print("\nRunning refutation tests...")

        refutation_results = run_refutation_tests(
            model,
            identified_estimand,
            estimate,
        )

        result_row.update(refutation_results)

        # Sensitivity analysis
        print("\nRunning sensitivity analysis...")

        sensitivity_results = run_sensitivity_analysis(
            model,
            identified_estimand,
            estimate,
        )

        for sensitivity in sensitivity_results:

            strength = sensitivity["strength"]

            result_row[
                f"sensitivity_ate_{strength:.2f}"
            ] = sensitivity["new_ate"]

            result_row[
                f"sensitivity_status_{strength:.2f}"
            ] = sensitivity["status"]

            result_row[
                f"sensitivity_detail_{strength:.2f}"
            ] = sensitivity["result"]

            all_sensitivity_results.append({
                "treatment": treatment,
                **sensitivity,
            })

        all_results.append(result_row)

    results_df = pd.DataFrame(all_results)

    sensitivity_df = pd.DataFrame(
        all_sensitivity_results
    )

    return results_df, sensitivity_df


# ============================================================
# 8. SAVE RESULTS
# ============================================================

def save_results(
    results_df,
    sensitivity_df,
):

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # Main results
    main_path = (
        OUTPUT_DIR / "causal_inference_results.csv"
    )

    results_df.to_csv(
        main_path,
        index=False
    )

    # Sensitivity results in long format
    sensitivity_path = (
        OUTPUT_DIR / "sensitivity_analysis_results.csv"
    )

    sensitivity_df.to_csv(
        sensitivity_path,
        index=False
    )

    print("\n" + "=" * 65)
    print("ANALYSIS COMPLETED")
    print("=" * 65)

    print(f"\nMain results saved to:\n{main_path}")

    print(
        f"\nSensitivity results saved to:\n{sensitivity_path}"
    )

    print("\nATE SUMMARY:")

    print(
        results_df[
            [
                "treatment",
                "ate",
                "ci_lower",
                "ci_upper",
                "ate_percentage_points",
            ]
        ].to_string(index=False)
    )

    print("\nSENSITIVITY SUMMARY:")

    print(
        sensitivity_df[
            [
                "treatment",
                "strength",
                "original_ate",
                "new_ate",
                "status",
            ]
        ].to_string(index=False)
    )

    # Check zero and missing sensitivity estimates
    zero_results = sensitivity_df[
        (sensitivity_df["new_ate"] == 0)
        & (sensitivity_df["status"] == "SUCCESS")
    ]

    missing_results = sensitivity_df[
        sensitivity_df["new_ate"].isna()
    ]

    print("\nDIAGNOSTIC CHECKS:")

    print(
        "Successful sensitivity estimates equal to zero:",
        len(zero_results)
    )

    print(
        "Missing or non-scalar sensitivity estimates:",
        len(missing_results)
    )

    if len(zero_results) > 0:

        print("\nZero estimates requiring inspection:")

        print(
            zero_results[
                ["treatment", "strength", "new_ate"]
            ].to_string(index=False)
        )


# ============================================================
# 9. MAIN EXECUTION
# ============================================================

def main():

    np.random.seed(RANDOM_SEED)

    print("=" * 65)
    print("CUSTOMER CHURN CAUSAL INFERENCE ANALYSIS")
    print("=" * 65)

    # Load and preprocess data
    df = load_data(DATA_PATH)

    # Run all analyses
    results_df, sensitivity_df = run_analysis(df)

    # Save results
    save_results(
        results_df,
        sensitivity_df,
    )

    return results_df, sensitivity_df


# ============================================================
# 10. RUN
# ============================================================

results_df, sensitivity_df = main()
